In [2]:
import spacy
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from transformers import AutoModelForQuestionAnswering
import torch

### Required Models:

In [3]:
# Load spaCy model for sentence segmentation + NER
# e.g. "en_core_web_sm" or a larger model for better performance
nlp = spacy.load("en_core_web_sm")

# MNLI model for entailment checking
entailment_model_name = "facebook/bart-large-mnli"
entailment_tokenizer = AutoTokenizer.from_pretrained(entailment_model_name)
entailment_model = AutoModelForSequenceClassification.from_pretrained(entailment_model_name)

# QA model (e.g., for ambiguous claim verification)
qa_model_name = "distilbert-base-cased-distilled-squad"
qa_pipeline = pipeline("question-answering", model=qa_model_name, tokenizer=qa_model_name, device=1)


Device set to use mps:0


### Utility functions:

In [4]:
def extract_claims_spacy(text):
    """
    Split text into sentences using spaCy and treat each as a 'claim.'
    """
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    return sentences

def ner_compare(generated_claim, source_text):
    """
    Very naive named-entity comparison:
    - Extract named entities in the claim and in the source text.
    - If an entity is not present at all in the source snippet or
      there's a mismatch, flag or log it for further checking.
    """
    claim_doc = nlp(generated_claim)
    source_doc = nlp(source_text)
    
    claim_ents = {(ent.text, ent.label_) for ent in claim_doc.ents}
    source_ents = {(ent.text, ent.label_) for ent in source_doc.ents}
    
    # Here we'll just see if the sets of entities have significant mismatch
    # This is a simplistic approach (e.g. partial overlaps might matter).
    mismatch = claim_ents - source_ents
    return mismatch

def simple_keyword_retrieval(claim, source_document, window=100):
    """
    Naive 'retrieval' approach: find the claim's keywords in the source doc,
    return a snippet around the first occurrence. In a real system, you would
    likely do more sophisticated chunking / indexing.
    """
    keywords = claim.split()[:3]  # Very naive: just first 3 words
    snippet = source_document
    for kw in keywords:
        idx = snippet.lower().find(kw.lower())
        if idx != -1:
            start = max(0, idx - window)
            end = min(len(snippet), idx + window)
            return snippet[start:end]
    # Fallback: return entire doc if no keyword found
    return source_document

def compute_entailment_probability(premise, hypothesis):
    """
    Returns the 'entailment' probability using an MNLI model:
      - premise = source snippet
      - hypothesis = claim
    """
    inputs = entailment_tokenizer.encode_plus(premise, hypothesis, return_tensors='pt')
    with torch.no_grad():
        logits = entailment_model(**inputs).logits
    # MNLI has 3 labels: 0 = entailment (in some models), 1 = neutral, 2 = contradiction
    # or 0 = contradiction, 1 = neutral, 2 = entailment, depending on the model
    # Check model documentation or config.id2label to confirm. For bart-large-mnli:
    # label order is [contradiction, neutral, entailment].
    probs = torch.softmax(logits, dim=-1).numpy()[0]
    contradiction_prob = probs[0]
    neutral_prob       = probs[1]
    entailment_prob    = probs[2]
    return float(entailment_prob)

def qa_validation_check(claim, snippet, threshold=0.5):
    """
    Attempt to validate ambiguous claims by turning them into a question
    and seeing if the QA model can produce a confident answer from the snippet.
    
    This is extremely naive: a real system would need specialized question generation
    or deeper claim analysis. For demonstration, we just take the claim as a 'question'.
    """
    if len(claim.split()) < 5:
        # Skip trivial short claims
        return {"is_ambiguous": False, "answer": None, "score": 1.0}
    
    # Attempt to treat the entire claim as a question
    # In practice, you'd parse the claim or rephrase it.
    question = claim
    try:
        result = qa_pipeline(question=question, context=snippet)
        # If the QA pipeline's answer is "unanswerable" or low confidence, we treat it as ambiguous
        if result["score"] < threshold or result["answer"].lower() in ["", "unanswerable"]:
            return {"is_ambiguous": True, "answer": result["answer"], "score": result["score"]}
        else:
            return {"is_ambiguous": False, "answer": result["answer"], "score": result["score"]}
    except:
        # If QA fails, treat it as ambiguous
        return {"is_ambiguous": True, "answer": None, "score": 0.0}


### Fact checking pipeline:

In [5]:
def fact_check_pipeline(generated_text, source_document, entailment_threshold=0.7):
    """
    Full pipeline that:
      1. Extracts claims from generated_text
      2. Performs naive NER checks
      3. For each claim:
          a) Retrieve snippet
          b) Compute entailment
          c) QA check for ambiguous claims
      4. Flags low-entailment claims as potential hallucinations
    """
    # 1) Claim extraction
    claims = extract_claims_spacy(generated_text)
    
    results = []
    
    for claim in claims:
        # 2) Naive NER check (compare with entire source for now)
        mismatch_entities = ner_compare(claim, source_document)
        
        # 3a) Retrieve snippet
        snippet = simple_keyword_retrieval(claim, source_document)
        
        # 3b) Compute entailment
        entail_prob = compute_entailment_probability(snippet, claim)
        
        # 3c) QA check
        qa_check = qa_validation_check(claim, snippet)
        
        # 4) Flag hallucinations
        # If entailment < threshold, we suspect hallucination
        is_hallucination = entail_prob < entailment_threshold
        
        claim_result = {
            "claim": claim,
            "snippet": snippet,
            "entailment_probability": entail_prob,
            "mismatched_entities": list(mismatch_entities),
            "qa_check": qa_check,
            "is_hallucination": is_hallucination
        }
        results.append(claim_result)
    
    return results

### Use case:

In [6]:
import pandas as pd

In [7]:
df=pd.read_csv("news_app_dataset.csv")

In [8]:
df

,query,text,topic,summary,news
0,Trump,"FloppySlapper: If you voted for Trump, this is...",President Trump Decisions and Policies,This text appears to be a collection of posts ...,Title: Enthusiasm for Trump's Policies Continu...
1,Ukraine war,Trump says Ukraine 'should never have started ...,Ukraine War,This conversation appears to be discussing the...,Title: Deepening Concerns Over Trump's Alleged...
2,Ukraine war,Trump says Ukraine 'should never have started ...,Political Analysis - Ukraine Conflict,It appears that this conversation revolves aro...,Title: Deepening Concerns Over Trump's Alleged...
3,LLM,AdventurousMistake72: What are people running ...,Applying to Law Schools (LLM),Hello! It seems like you have a mix of persona...,Title: Stanford vs. Georgetown: The Tough Choi...
4,Diamonds,NoopKit: Why do people still buy diamonds? ver...,Diamond Market Scarcity and Pricing,The discussion revolves around the perceived s...,Exploring the Shift in the Diamond Market: Fro...
5,Bob Dylan,Pyromania1983: [Serious] What specifically is ...,Bob Dylan Personality,"From the conversation, it appears that Bob Dyl...",Title: Unpacking the Enigma That Is Bob Dylan:...
6,Timothee Chalamet,vitkurd: Timothee Chalamet with Kylie Jenner a...,Movie Review: Timothee Chalamet Career Analysis,The conversation appears to be about a video m...,Headline: Timothée Chalamet's Rising Star: A R...
7,Van Gogh,Snorlax_Cuddles: When did people actually real...,Vincent van Gogh,It looks like you've shared a collection of co...,Title: A Timeless Gaze: The Enduring Impact of...
8,Economy,Rebelliousdefender: I hate the lies about the ...,Economy and Business,The text appears to be a collection of comment...,Title: America's Economic and Political Landsc...
9,Corona virus,mostrandomguy: Medical professionals around th...,COVID-19 Testing Issues in the US,It appears you've shared a discussion thread a...,Title: U.S. Struggles to Overcome Testing Shor...


In [9]:
generated_text=df["news"][7]

In [10]:
source_document=df["text"][7]

In [11]:
if __name__ == "__main__":
    
    pipeline_results = fact_check_pipeline(generated_text, source_document)
    
    for res in pipeline_results:
        print("Claim:", res["claim"])
        print("Entailment Probability:", res["entailment_probability"])
        print("Mismatched Entities:", res["mismatched_entities"])
        print("QA Check:", res["qa_check"])
        print("Is Hallucination?", res["is_hallucination"])
        print("-" * 60)

: 

: 